In [ ]:
!mkdir data

In [ ]:
import pandas as pd
import gradio as gr
import joblib

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("/content/data/Housing.csv")

In [ ]:
print("Dataset loaded successfully!")
print(df.head())


Dataset loaded successfully!
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  


In [ ]:
print("\nDataset Shape:")
print(df.shape)


Dataset Shape:
(545, 13)


In [ ]:
print("\nColumns:")
print(df.columns.tolist())


Columns:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']


In [ ]:
print(df.isnull().sum())

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64


In [ ]:

df = df.dropna()

print("\nAfter removing missing values:")
print(df.shape)


After removing missing values:
(545, 13)


In [ ]:
X = df.drop("price", axis=1)

y = df["price"]

In [ ]:
categorical_features = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea",
    "furnishingstatus"
]

In [ ]:
numerical_features = [
    "area",
    "bedrooms",
    "bathrooms",
    "stories",
    "parking"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [ ]:
model = LinearRegression()

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regression", model)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


In [ ]:
print("\nTraining Linear Regression model...")

pipeline.fit(X_train, y_train)

print("Model trained successfully!")


Training Linear Regression model...
Model trained successfully!


In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)


print("\n==============================")
print("MODEL EVALUATION")
print("==============================")

print(f"MAE  : ₹{mae:,.2f}")
print(f"MSE  : ₹{mse:,.2f}")
print(f"RMSE : ₹{rmse:,.2f}")
print(f"R²   : {r2:.4f}")


MODEL EVALUATION
MAE  : ₹970,043.40
MSE  : ₹1,754,318,687,330.70
RMSE : ₹1,324,506.96
R²   : 0.6529


In [ ]:
joblib.dump(
    pipeline,
    "house_price_regression_model.pkl"
)

print("\nModel saved successfully!")


Model saved successfully!


In [ ]:
def predict_price(
    area,
    bedrooms,
    bathrooms,
    stories,
    mainroad,
    guestroom,
    basement,
    hotwaterheating,
    airconditioning,
    parking,
    prefarea,
    furnishingstatus
):

    # Create dataframe
    input_data = pd.DataFrame({

        "area": [area],

        "bedrooms": [bedrooms],

        "bathrooms": [bathrooms],

        "stories": [stories],

        "mainroad": [mainroad],

        "guestroom": [guestroom],

        "basement": [basement],

        "hotwaterheating": [hotwaterheating],

        "airconditioning": [airconditioning],

        "parking": [parking],

        "prefarea": [prefarea],

        "furnishingstatus": [furnishingstatus]
    })


    # Predict
    prediction = pipeline.predict(input_data)[0]


    return f"₹ {prediction:,.0f}"

In [ ]:
with gr.Blocks(
    title="House Price Prediction"
) as demo:

    gr.Markdown(
        """
        # 🏠 House Price Prediction

        ### Linear Regression Model

        Enter the details of your house below to get
        an estimated house price.
        """
    )


    # -----------------------------------------------------
    # NUMERICAL INPUTS
    # -----------------------------------------------------

    gr.Markdown("### 🏡 Property Details")

    with gr.Row():

        area = gr.Number(
            label="Area (sq ft)",
            value=5000
        )

        bedrooms = gr.Number(
            label="Bedrooms",
            value=3,
            precision=0
        )

        bathrooms = gr.Number(
            label="Bathrooms",
            value=2,
            precision=0
        )


    with gr.Row():

        stories = gr.Number(
            label="Stories",
            value=2,
            precision=0
        )

        parking = gr.Number(
            label="Parking Spaces",
            value=1,
            precision=0
        )


    # -----------------------------------------------------
    # YES / NO FEATURES
    # -----------------------------------------------------

    gr.Markdown("### ⚙️ House Features")

    with gr.Row():

        mainroad = gr.Dropdown(
            choices=["yes", "no"],
            value="yes",
            label="Main Road"
        )

        guestroom = gr.Dropdown(
            choices=["yes", "no"],
            value="no",
            label="Guest Room"
        )

        basement = gr.Dropdown(
            choices=["yes", "no"],
            value="no",
            label="Basement"
        )


    with gr.Row():

        hotwaterheating = gr.Dropdown(
            choices=["yes", "no"],
            value="no",
            label="Hot Water Heating"
        )

        airconditioning = gr.Dropdown(
            choices=["yes", "no"],
            value="yes",
            label="Air Conditioning"
        )

        prefarea = gr.Dropdown(
            choices=["yes", "no"],
            value="yes",
            label="Preferred Area"
        )


    # -----------------------------------------------------
    # FURNISHING
    # -----------------------------------------------------

    furnishingstatus = gr.Dropdown(
        choices=[
            "furnished",
            "semi-furnished",
            "unfurnished"
        ],
        value="semi-furnished",
        label="Furnishing Status"
    )


    # -----------------------------------------------------
    # BUTTON
    # -----------------------------------------------------

    predict_button = gr.Button(
        "🔮 Predict House Price",
        variant="primary"
    )


    # -----------------------------------------------------
    # OUTPUT
    # -----------------------------------------------------

    output = gr.Textbox(
        label="Predicted House Price",
        interactive=False
    )


    # -----------------------------------------------------
    # BUTTON FUNCTION
    # -----------------------------------------------------

    predict_button.click(

        fn=predict_price,

        inputs=[
            area,
            bedrooms,
            bathrooms,
            stories,
            mainroad,
            guestroom,
            basement,
            hotwaterheating,
            airconditioning,
            parking,
            prefarea,
            furnishingstatus
        ],

        outputs=output
    )


    # -----------------------------------------------------
    # MODEL INFORMATION
    # -----------------------------------------------------

    gr.Markdown(
        """
        ### 📊 Model Information

        **Algorithm:** Linear Regression

        **Target:** House Price

        **Input Features:** Area, bedrooms, bathrooms,
        stories, parking, road access, guest room,
        basement, heating, air conditioning, preferred
        area and furnishing status.

        ⚠️ This prediction is for educational purposes.
        """
    )


# =========================================================
# 16. LAUNCH
# =========================================================

if __name__ == "__main__":

    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1b7a0b62fa1aa556ed.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
